In [231]:
import altair as alt
import pandas as pd

In [232]:
from pathlib import Path
PROJECT_DIR = Path()

combined_data_df = pd.read_csv(PROJECT_DIR/"data"/"cleaned"/"all_data_combined.csv")

# TODO: move this to data/clean_data.py
combined_data_df["Years colonized"] = (
    combined_data_df.groupby("Entity")["Years colonized"].transform("max")
)

In [233]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [234]:
rights_cols = ["Legal status of homosexuality", "Same-sex marriage",
               "Censorship of LGBT+ issues", "Gender-affirming care",
               "Right to change legal gender"]

score_maps = {
    "Legal status of homosexuality": {"Illegal": 0, "Legal": 2},
    "Same-sex marriage":             {"Legal": 2, "Banned": 0, "Civil union or other partnership": 1, "Unrecognized": 0.5, "Varies by region": 0.5, "Unregistered cohabitation": 0.5, "Foreign same-sex marriages recognized only": 0.5, "Ambiguous": 0},
    "Censorship of LGBT+ issues":    {"Imprisonment as punishment": 0, "State-enforced": 0, "No censorship": 2, "Varies by region": 0.5, "Fine as punishment": 0, "Death penalty as punishment": 0, "Ambiguous": 0},
    "Gender-affirming care":         {"Banned": 0, "Restricted": 0, "Legal": 2, "Legal, but restricted for minors": 1, "Legal, but banned for minors": 0.5, "Varies by region": 0.5, "Ambiguous": 0},
    "Right to change legal gender":  {"Legal, surgery required": 1, "Legal, medical diagnosis required": 1, "Illegal": 0, "Legal, no restrictions": 2, "Varies by region": 0.5, "Ambiguous": 0},
}
for col, m in score_maps.items():
    combined_data_df[col + " (score)"] = combined_data_df[col].map(m)

score_cols = [c + " (score)" for c in rights_cols]
combined_data_df["Rights index"] = combined_data_df[score_cols].sum(axis=1, min_count=len(score_cols))
combined_data_df["Colonizer"] = combined_data_df["Colonizer"].fillna("Unknown")

In [235]:
#TODO: move this into data/clean_data.py to add it to the CSV
import pycountry

def to_numeric(code):
    try:
        return int(pycountry.countries.get(alpha_3=code).numeric)
    except Exception:
        return None

combined_data_df["id"] = combined_data_df["Code"].map(to_numeric)

In [236]:
alt.themes.enable('fivethirtyeight')

ThemeRegistry.enable('fivethirtyeight')

In [237]:
from vega_datasets import data

countries = alt.topo_feature(data.world_110m.url, "countries")
df_2025 = combined_data_df[combined_data_df["Year"] == 2025]

choropleth = (
    alt.Chart(countries)
    .mark_geoshape(stroke="white")
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(df_2025, "id", ["Entity", "Legal status of homosexuality"]),
    )
    .encode(
        color=alt.Color("Legal status of homosexuality:N",
                        title="Legal Status"),
        tooltip=["Entity:N", "Legal status of homosexuality:N"],
    )
    .properties(width=650, height=350, title="Legal Status of Homosexuality in 2025")
)
choropleth.save(PROJECT_DIR/"visualizations"/"2.png")

In [238]:
df_2000 = combined_data_df[combined_data_df["Year"] == 2000]

choropleth = (
    alt.Chart(countries)
    .mark_geoshape(stroke="white")
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(df_2000, "id", ["Entity", "Legal status of homosexuality"]),
    )
    .encode(
        color=alt.Color("Legal status of homosexuality:N",
                        title="Legal Status"),
        tooltip=["Entity:N", "Legal status of homosexuality:N"],
    )
    .properties(width=650, height=350, title="Legal Status of Homosexuality in 2000")
)
choropleth
choropleth.save(PROJECT_DIR/"visualizations"/"1.png")

In [ ]:
df_1975 = combined_data_df[combined_data_df["Year"] == 1975]

choropleth = (
    alt.Chart(countries)
    .mark_geoshape(stroke="white")
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(df_1975, "id", ["Entity", "Legal status of homosexuality"]),
    )
    .encode(
        color=alt.Color("Legal status of homosexuality:N",
                        title="Legal Status"),
        tooltip=["Entity:N", "Legal status of homosexuality:N"],
    )
    .properties(width=650, height=350, title="Legal Status of Homosexuality in 1975")
)
choropleth

In [240]:
alt.themes.enable('dark')

ThemeRegistry.enable('dark')

In [241]:
df_2000 = combined_data_df[combined_data_df["Year"] == 2000]

choropleth = (
    alt.Chart(countries)
    .mark_geoshape(stroke="white")
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(df_2000, "id", ["Entity", "Same-sex marriage"]),
    )
    .encode(
        color=alt.Color("Same-sex marriage:N",
                        title="Legal Status"),
        tooltip=["Entity:N", "Same-sex marriage:N"],
    )
    .properties(width=650, height=350, title="Same-sex marriage in 2000")
)
choropleth.save(PROJECT_DIR/"visualizations"/"3.png")

In [242]:
df_2025 = combined_data_df[combined_data_df["Year"] == 2025]

choropleth = (
    alt.Chart(countries)
    .mark_geoshape(stroke="white")
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(df_2025, "id", ["Entity", "Same-sex marriage"]),
    )
    .encode(
        color=alt.Color("Same-sex marriage:N",
                        title="Legal Status"),
        tooltip=["Entity:N", "Same-sex marriage:N"],
    )
    .properties(width=650, height=350, title="Same-sex marriage in 2025")
)
choropleth.save(PROJECT_DIR/"visualizations"/"4.png")

In [243]:
df_2025 = combined_data_df[combined_data_df["Year"] == 2025]

choropleth = (
    alt.Chart(countries)
    .mark_geoshape(stroke="white")
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(df_2025, "id", ["Entity", "Years colonized"]),
    )
    .encode(
        color=alt.Color("Years colonized:Q"),
        tooltip=["Entity:N", "Years colonized:Q"],
    )
    .properties(width=650, height=350, title="Number of years colonized")
)
choropleth.save(PROJECT_DIR/"visualizations"/"5.png")

In [244]:
alt.themes.enable('fivethirtyeight')

ThemeRegistry.enable('fivethirtyeight')

In [245]:
equaldex_df = pd.read_csv(PROJECT_DIR/"data"/"cleaned"/"equaldex_with_colonial_history.csv")

In [246]:
bars = (
    alt.Chart(equaldex_df)
    .mark_bar()
    .encode(
        x=alt.X("mean(EI):Q", title="Mean Equality Index"),
        y=alt.Y("Longest colonizer:N", sort="-x"),
        tooltip=["Longest colonizer:N", "mean(EI):Q", "count():Q"],
    )
    .properties(width=650, height=350, title="Mean Equality Index of countries with the same longest colonizer")
)

bars.save(PROJECT_DIR/"visualizations"/"6.png")

In [247]:
legal_status_bars = (
    alt.Chart(combined_data_df.dropna(subset="Legal status of homosexuality"))
    .mark_bar()
    .encode(
        x=alt.X("count():Q", stack="normalize", title="Share of countries"),
        y=alt.Y("Longest colonizer:N"),
        color="Legal status of homosexuality:N",
        tooltip=["Longest colonizer", "Legal status of homosexuality", "count()"],
    )
    .properties(title="Legal status of homosexaulity in colonies across time")
)
legal_status_bars.save(PROJECT_DIR/"visualizations"/"7.png")

In [ ]:
scatter = (
    alt.Chart(equaldex_df.dropna(subset=["EI Legal", "EI PO"]))
    .mark_circle(size=70, opacity=0.7)
    .encode(
        x=alt.X("EI Legal:Q", title="Legal index"),
        y=alt.Y("EI PO:Q", title="Public opinion index"),
        color="Longest colonizer:N",
        tooltip=["Name", "EI", "EI Legal", "EI PO", "Longest colonizer"],
    )
    .properties(title="Countries' LGBT Equality Indexes")
)

diagonal = alt.Chart(pd.DataFrame({"x": [0, 100], "y": [0, 100]})).mark_line(
    strokeDash=[4, 4], color="gray").encode(x="x", y="y")

(scatter + diagonal)

In [261]:
scatter
scatter.save(PROJECT_DIR/"visualizations"/"8.png")

In [256]:
ei_vs_years_colonized_scatter = (
    alt.Chart(equaldex_df)
    .mark_circle(size=70, opacity=0.7)
    .encode(
        x=alt.X("Years colonized:Q", title="Years colonized"),
        y=alt.Y("EI:Q", title="Equality index"),
        color="Longest colonizer:N",
        tooltip=["Name:N", "EI:Q", "Years colonized:Q"],
    )
    .properties(title="Countries' Overall LGBT Equality Index vs. Years colonized")

)

ei_vs_years_colonized_scatter.save(PROJECT_DIR/"visualizations"/"9.png")

In [251]:
box_plot = (
    alt.Chart(equaldex_df)
    .mark_boxplot()
    .encode(
        x="Longest colonizer:N", 
        y=alt.Y("EI:Q", title="Equality Index"), 
        color="Longest colonizer:N")
    .properties(title="LGBT Equality Index by longest colonizer")
)
box_plot.save(PROJECT_DIR/"visualizations"/"10.png")

In [ ]:
(
    alt.Chart(combined_data_df.dropna(subset=["Rights index"]))
    .mark_line()
    .encode(
        x="Year:O",
        y=alt.Y("mean(Rights index):Q", title="Mean rights index"),
        color="Colonizer:N",
    )
)

In [262]:
heatmap = (
    alt.Chart(equaldex_df)
    .mark_rect()
    .encode(
        x=alt.X("Years colonized:Q"),
        y=alt.Y("EI:Q", title="Equality index (EI)"),
        color=alt.Color("EI:Q"),
        tooltip=["Years colonized:Q", "EI:Q", "count():Q"],
    )
    .properties(title="Years colonized vs. equality index")
)
heatmap.save(PROJECT_DIR/"visualizations"/"11.png")

In [ ]:
(
    alt.Chart(combined_data_df)
    .mark_rect()
    .encode(
        x=alt.X("Longest colonizer:N", axis=alt.Axis(labelAngle=-30)),
        y="Years colonized:Q",
        color=alt.Color("mean(Rights index):Q"),
        tooltip=["Years colonized:Q", "mean(Rights index):Q", "count():Q"],
    )
)